# آزمایش تعاملی استدلال ساختارمند در مدل‌های زبانی

این نوت‌بوک برای اجرای پروژه در **Google Colab** آماده شده است. می‌توانید تعداد سؤال‌ها و تنظیمات اجرا را انتخاب کنید، نتایج را در جدول و نمودار ببینید، پاسخ هر سؤال و خروجی خام مدل را بررسی کنید و فایل نتایج را دریافت کنید.

**مدل:** `Qwen/Qwen2.5-0.5B-Instruct`  
**داده:** مجموعه آزمون GSM8K  
**روش‌ها:** Direct، CoT و ToT-inspired

نکته پژوهشی: روش سوم نسخه ساده‌شده و الهام‌گرفته از Tree of Thoughts است، نه پیاده‌سازی کامل جست‌وجوی درختی. تعداد توکن‌ها شاخص حجم خروجی است، نه هزینه مالی.

### شروع
در صورت دسترسی، از Runtime → Change runtime type گزینه GPU (T4) را انتخاب کنید. سپس سلول‌ها را از بالا به پایین اجرا کنید.

In [ ]:
import os, sys, json, subprocess, zipfile
from pathlib import Path
from datetime import datetime
import torch
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("محیط روی CPU است؛ اجرای مدل کندتر خواهد بود.")

In [ ]:
# دریافت آخرین کد پروژه
%cd /content
if not Path("/content/structured-reasoning-llm").exists():
    !git clone https://github.com/itsilsil/structured-reasoning-llm.git
%cd /content/structured-reasoning-llm

In [ ]:
# نصب وابستگی‌ها؛ PyTorch موجود در Colab حفظ می‌شود
!pip -q install "transformers>=4.45.0" "accelerate>=0.34.0" "datasets>=3.0.0" "sentencepiece>=0.2.0" "pandas>=2.2.0" "tqdm>=4.66.0" "ipywidgets>=8.1.0" "matplotlib>=3.8.0"
print("نصب وابستگی‌ها تمام شد.")

## ۱. اجرای آزمایش با تنظیمات دلخواه

برای بررسی اولیه ۵ سؤال را انتخاب کنید. هر اجرا در پوشه جداگانه ذخیره می‌شود و نتایج اجراهای قبلی بازنویسی نمی‌شوند.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd

sample_w = widgets.Dropdown(options=[("۵ سؤال",5),("۱۰ سؤال",10)], value=5, description="تعداد سؤال:")
seed_w = widgets.IntText(value=42, description="Seed:")
tokens_w = widgets.Dropdown(options=[("۱۲۸ توکن",128),("۲۵۶ توکن",256),("۵۱۲ توکن",512)], value=256, description="حداکثر توکن:")
candidates_w = widgets.Dropdown(options=[("۲ مسیر",2),("۳ مسیر",3)], value=2, description="تعداد مسیرها:")
run_btn = widgets.Button(description="اجرای آزمایش", button_style="success", icon="play")
run_out = widgets.Output()
last_run_dir = None

def run_experiment(_):
    global last_run_dir
    with run_out:
        clear_output(wait=True)
        run_name = datetime.now().strftime("run_%Y%m%d_%H%M%S")
        out_dir = Path("runs") / run_name
        cmd = [sys.executable,"src/experiment.py","--sample-size",str(sample_w.value),
               "--seed",str(seed_w.value),"--num-candidates",str(candidates_w.value),
               "--max-new-tokens",str(tokens_w.value),"--output-dir",str(out_dir)]
        print("در حال اجرا؛ لطفاً تا پایان سلول صبر کنید.\n")
        proc = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        print(proc.stdout)
        if proc.returncode != 0:
            print("اجرا با خطا متوقف شد. متن خطا را برای بررسی بفرستید.")
            return
        last_run_dir = out_dir
        print("اجرا تمام شد:", out_dir)
        display(pd.read_csv(out_dir / "summary.csv"))

run_btn.on_click(run_experiment)
display(widgets.VBox([sample_w,seed_w,tokens_w,candidates_w,run_btn,run_out]))

## ۲. نمودارها و جزئیات نتایج

پس از پایان آزمایش، این سلول را اجرا کنید.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
if last_run_dir is None:
    print("ابتدا آزمایش را اجرا کنید.")
else:
    summary = pd.read_csv(last_run_dir / "summary.csv")
    records = [json.loads(x) for x in (last_run_dir / "experiment_results.jsonl").read_text(encoding="utf-8").splitlines() if x.strip()]
    display(Markdown("### جدول مقایسه روش‌ها"))
    display(summary)
    for col,title,ylabel,mult in [
        ("accuracy","دقت پاسخ‌گویی","درصد",100),
        ("avg_generated_tokens","میانگین توکن تولیدشده","تعداد",1),
        ("avg_generation_calls","میانگین فراخوانی تولید","تعداد",1)]:
        fig,ax = plt.subplots(figsize=(7,4))
        ax.plot(summary["method"],summary[col]*mult,marker="o",linewidth=2)
        ax.set_title(title); ax.set_ylabel(ylabel); ax.grid(True,alpha=.3)
        if col == "accuracy": ax.set_ylim(0,100)
        plt.xticks(rotation=10); plt.tight_layout(); plt.show()
    display(Markdown("### جزئیات هر سؤال"))
    detail = pd.DataFrame(records)
    cols = ["dataset_index","question","method","gold_answer","predicted_answer","correct","generated_tokens","generation_calls"]
    display(detail[cols].sort_values(["dataset_index","method"]).reset_index(drop=True))
    display(Markdown("### خروجی خام مدل"))
    for rec in records:
        area = widgets.Textarea(value=str(rec.get("raw_output","")),layout=widgets.Layout(width="100%",height="150px"))
        acc = widgets.Accordion(children=[area])
        acc.set_title(0,f"سؤال {rec['dataset_index']} | {rec['method']} | درست: {rec['correct']}")
        display(acc)

## ۳. دریافت فایل نتایج

بسته ZIP شامل جدول خلاصه، خروجی خام مدل، تنظیمات اجرا و جزئیات سؤال‌هاست.

In [ ]:
from google.colab import files
if last_run_dir is None:
    print("ابتدا یک آزمایش اجرا کنید.")
else:
    detail = pd.DataFrame(records)
    detail.to_csv(last_run_dir / "question_details.csv",index=False,encoding="utf-8-sig")
    zip_path = Path("/content") / f"{last_run_dir.name}_results.zip"
    with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
        for name in ["summary.csv","experiment_results.jsonl","run_config.json","question_details.csv"]:
            z.write(last_run_dir / name,arcname=name)
    files.download(str(zip_path))

## نکات تفسیر علمی

- پاسخ‌هایی که قالب مشخص `Final answer:` ندارند، به‌عنوان پاسخ نادرست شمرده می‌شوند.
- روش ToT-inspired مسیرهای کاندید را تولید می‌کند و همان مدل یکی را انتخاب می‌کند؛ این پیاده‌سازی جست‌وجوی کامل درختی نیست.
- چون تولیدکننده و ارزیاب یک مدل هستند، احتمال سوگیری خودارزیابی وجود دارد.
- نتایج ۵ یا ۱۰ سؤال مقدماتی‌اند و برای نتیجه‌گیری آماری کافی نیستند.
- تعداد توکن‌های خروجی معادل زمان اجرا، مصرف انرژی یا هزینه مالی نیست.